# COMP8240 · SanText project update: every run in one notebook

**Before you start**
1. Check **Runtime → Change runtime type** says **T4 GPU**, then **Save**.
2. **Runtime → Run all**. If Colab warns the notebook wasn't written by Google, choose **Run anyway**.
3. Approve the **Google Drive** pop-up. Results are copied to `MyDrive/COMP8240_update`, so a disconnect loses nothing.
4. Keep this tab open, and click into it every 20–30 minutes so Colab doesn't disconnect.

| Step | What runs | Time |
|---|---|---|
| 0 | Setup and GPU check | 2 min |
| 1 | SST-2 download; GloVe (2.2 GB) starts downloading in the background | 1 min |
| 2 | Constructed dataset: collect → preprocess → 100-review annotation sample | 5 min |
| 3 | Fine-tuning smoke test, then the unsanitized baseline | 10 min |
| 4 | SanText sanitization at ε = 1, 2, 3 (the authors' code, unchanged) | 10–15 min |
| 5 | Fine-tuning on the sanitized data | 30 min |
| 6 | Results vs the paper, memory, examples → `core_results.zip` | 1 min |
| 7–9 | Optional: Dreaddit, SanText on the review dataset, full zip | 15–20 min |

**When Step 2 prints "Annotation sheet ready"**, start annotating (guidelines are just below Step 2). Everything else keeps running.

**When it finishes**, upload `comp8240_update_outputs.zip` in the chat (or `core_results.zip` if you stop after Step 6), or paste the printed summaries.

In [ ]:
# Step 0 · Setup and GPU check (about 2 minutes)
import os, sys, re, json, time, glob, shutil, subprocess
BASE = os.getcwd()  # /content on Colab
os.makedirs(f"{BASE}/logs", exist_ok=True)
os.makedirs(f"{BASE}/results", exist_ok=True)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime > Change runtime type > T4 GPU > Save, then Runtime > Run all again.")
print("GPU:", torch.cuda.get_device_name(0))

BACKUP = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    BACKUP = "/content/drive/MyDrive/COMP8240_update"
    os.makedirs(BACKUP, exist_ok=True)
    print("Drive backup folder:", BACKUP)
except Exception as e:
    print("No Drive backup; results stay in this session only:", e)

def backup(*paths):
    """Copy files or folders into the Drive backup folder (does nothing without Drive)."""
    if not BACKUP:
        return
    done = []
    for p in paths:
        if not os.path.exists(p):
            continue
        dest = os.path.join(BACKUP, os.path.relpath(p, BASE))
        if os.path.isdir(p):
            shutil.copytree(p, dest, dirs_exist_ok=True)
        else:
            os.makedirs(os.path.dirname(dest), exist_ok=True)
            shutil.copy2(p, dest)
        done.append(os.path.relpath(p, BASE))
    print("backed up to Drive:", ", ".join(done))

if not os.path.isdir(f"{BASE}/SanText"):
    !git clone -q https://github.com/xiangyue9607/SanText.git {BASE}/SanText
!pip -q install google-play-scraper
!cd {BASE}/SanText && git log -1 --format="SanText code: commit %h, %cd" --date=short
!python -c "import sys, torch, transformers, spacy, sklearn, numpy; print('python', sys.version.split()[0], '| torch', torch.__version__, '| transformers', transformers.__version__, '| spacy', spacy.__version__, '| scikit-learn', sklearn.__version__, '| numpy', numpy.__version__)"
!nproc
!free -g | head -2

In [ ]:
%%writefile finetune_sanitized.py
#!/usr/bin/env python3
"""
finetune_sanitized.py: fine-tune BERT on (sanitized) text and report dev-set utility.

Why this script exists
  SanText's released fine-tuning code (run_glue.py + dataset_glue.py) imports
  transformers 3.3.1 module paths such as `transformers.tokenization_bart`,
  which were removed in transformers 4.0, so it no longer runs on a current
  install. This script re-implements the same fine-tuning step with the paper's
  settings (Yue et al., 2021, Sec. 5.1, SST-2/QNLI):
      bert-base-uncased, max_seq_length 128, batch size 64, learning rate 2e-5,
      3 epochs, AdamW without weight decay, linear decay without warmup,
      gradient clipping 1.0, seed 42.
  One deliberate difference: each batch is padded to its longest example rather
  than to 128 tokens. Padding is masked out of attention, so the model computes
  the same thing, but short-sentence data such as SST-2 trains several times faster.

Input
  --data_dir must contain train.tsv and dev.tsv with a header row, as written by
  run_SanText.py (or the original GLUE files):
    single sentence (SST-2 style):  sentence <TAB> label
    sentence pair  (QNLI style):    index <TAB> text_a <TAB> text_b <TAB> label
  The format is detected from the header (2 columns = single, 4 or more = pair).

Output
  results/<run_name>/metrics.json         settings, environment, per-epoch and final dev metrics
  results/<run_name>/dev_predictions.tsv  gold and predicted label for every dev row

Examples
  # 1-minute smoke test (run this first on a new machine)
  python finetune_sanitized.py --data_dir data/SST-2 --max_train_samples 512 --epochs 1 --run_name smoke_test
  # unsanitized baseline (paper: 0.9251 accuracy)
  python finetune_sanitized.py --data_dir data/SST-2 --run_name SST-2_unsanitized --fp16
  # sanitized data written by run_SanText.py
  python finetune_sanitized.py --data_dir output_SanText_glove/SST-2/eps_3.00 --fp16
  # long texts (e.g. Dreaddit): longer inputs, same effective batch of 64 in two halves
  python finetune_sanitized.py --data_dir new_data/dreaddit --max_seq_length 256 --grad_accum 2 --fp16
"""
import argparse
import json
import math
import os
import platform
import random
import time
from contextlib import nullcontext

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import numpy as np
import torch
from torch.utils.data import DataLoader

import transformers
from transformers import AutoModelForSequenceClassification, AutoTokenizer


def parse_args():
    p = argparse.ArgumentParser(description="Fine-tune BERT on (sanitized) TSV data and report dev-set utility.")
    p.add_argument("--data_dir", required=True, help="folder containing train.tsv and dev.tsv")
    p.add_argument("--run_name", default=None, help="results folder name (default: built from data_dir)")
    p.add_argument("--output_root", default="results")
    p.add_argument("--model_name", default="bert-base-uncased")
    p.add_argument("--max_seq_length", type=int, default=128)
    p.add_argument("--batch_size", type=int, default=64, help="effective batch size per optimizer step")
    p.add_argument("--grad_accum", type=int, default=1,
                   help="split each batch into this many micro-batches to save GPU memory (effective batch unchanged)")
    p.add_argument("--eval_batch_size", type=int, default=128)
    p.add_argument("--learning_rate", type=float, default=2e-5)
    p.add_argument("--epochs", type=int, default=3)
    p.add_argument("--warmup_ratio", type=float, default=0.0)
    p.add_argument("--weight_decay", type=float, default=0.0)
    p.add_argument("--max_grad_norm", type=float, default=1.0)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--fp16", action="store_true", help="mixed precision on GPU (faster on T4/P100-class GPUs)")
    p.add_argument("--format", choices=["auto", "single", "pair"], default="auto")
    p.add_argument("--max_train_samples", type=int, default=None, help="use only the first N training rows (smoke tests)")
    p.add_argument("--log_every", type=int, default=200, help="print training loss every N optimizer steps")
    p.add_argument("--save_model", action="store_true", help="also save the fine-tuned model (about 440 MB)")
    args = p.parse_args()
    if args.batch_size % args.grad_accum != 0:
        p.error("--batch_size must be divisible by --grad_accum")
    return args


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def read_tsv(path, fmt):
    """Return (format, texts_a, texts_b or None, labels, skipped_rows)."""
    texts_a, texts_b, labels, skipped = [], [], [], 0
    with open(path, encoding="utf-8") as f:
        header = f.readline().rstrip("\n").split("\t")
        if fmt == "auto":
            fmt = "single" if len(header) == 2 else "pair"
        for line in f:
            cols = line.rstrip("\n").split("\t")
            if fmt == "single":
                text, label = "\t".join(cols[:-1]).strip(), cols[-1].strip()
                if len(cols) < 2 or not text or not label:
                    skipped += 1
                    continue
                texts_a.append(text)
            else:
                if len(cols) < 4 or not cols[-1].strip():
                    skipped += 1
                    continue
                texts_a.append(cols[1])
                texts_b.append(cols[2])
                label = cols[-1].strip()
            labels.append(label)
    return fmt, texts_a, (texts_b if fmt == "pair" else None), labels, skipped


def build_label_map(labels):
    def key(x):
        return (0, int(x), "") if x.lstrip("-").isdigit() else (1, 0, x)
    return {lab: i for i, lab in enumerate(sorted(set(labels), key=key))}


def make_examples(texts_a, texts_b, labels, label_map, split_name):
    unknown = sorted(set(labels) - set(label_map))
    if unknown:
        raise ValueError(f"{split_name}.tsv has labels not seen in train.tsv: {unknown[:5]}")
    b = texts_b if texts_b is not None else [None] * len(texts_a)
    return [(a, bb, label_map[lab]) for a, bb, lab in zip(texts_a, b, labels)]


def make_collate(tokenizer, max_len):
    def collate(batch):
        a = [ex[0] for ex in batch]
        b = [ex[1] for ex in batch] if batch[0][1] is not None else None
        enc = tokenizer(a, b, truncation=True, max_length=max_len, padding=True, return_tensors="pt")
        enc["labels"] = torch.tensor([ex[2] for ex in batch], dtype=torch.long)
        return enc
    return collate


def make_scaler(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):  # older torch
        return torch.cuda.amp.GradScaler(enabled=enabled)


def amp(enabled):
    """Mixed-precision context on GPU; a no-op otherwise."""
    return torch.autocast(device_type="cuda", dtype=torch.float16) if enabled else nullcontext()


def compute_metrics(gold, pred, n_labels):
    gold, pred = np.asarray(gold), np.asarray(pred)
    f1s = []
    for c in range(n_labels):
        tp = int(((pred == c) & (gold == c)).sum())
        fp = int(((pred == c) & (gold != c)).sum())
        fn = int(((pred != c) & (gold == c)).sum())
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec = tp / (tp + fn) if tp + fn else 0.0
        f1s.append(2 * prec * rec / (prec + rec) if prec + rec else 0.0)
    out = {"accuracy": float((gold == pred).mean()), "macro_f1": float(np.mean(f1s))}
    if n_labels == 2:
        out["f1_label_index_1"] = float(f1s[1])  # positive class for SST-2, "stress" for Dreaddit
    return out


@torch.no_grad()
def evaluate(model, loader, device, use_fp16, n_labels):
    model.eval()
    preds, golds = [], []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with amp(use_fp16):
            logits = model(**{k: v for k, v in batch.items() if k != "labels"}).logits
        preds.extend(logits.argmax(dim=-1).tolist())
        golds.extend(batch["labels"].tolist())
    model.train()
    return compute_metrics(golds, preds, n_labels), preds


def main():
    args = parse_args()
    set_seed(args.seed)
    t_start = time.time()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_fp16 = args.fp16 and device.type == "cuda"
    if args.fp16 and not use_fp16:
        print("note: --fp16 ignored because no GPU is available")
    gpu_name = torch.cuda.get_device_name(0) if device.type == "cuda" else None

    # ---- data -------------------------------------------------------------
    fmt, tr_a, tr_b, tr_y, tr_skip = read_tsv(os.path.join(args.data_dir, "train.tsv"), args.format)
    _, dv_a, dv_b, dv_y, dv_skip = read_tsv(os.path.join(args.data_dir, "dev.tsv"), fmt)
    if args.max_train_samples:
        tr_a, tr_y = tr_a[: args.max_train_samples], tr_y[: args.max_train_samples]
        tr_b = tr_b[: args.max_train_samples] if tr_b is not None else None
    label_map = build_label_map(tr_y)
    n_labels = len(label_map)
    train_ex = make_examples(tr_a, tr_b, tr_y, label_map, "train")
    dev_ex = make_examples(dv_a, dv_b, dv_y, label_map, "dev")
    print(f"data: {args.data_dir} | format={fmt} | train={len(train_ex)} dev={len(dev_ex)} "
          f"| skipped rows train={tr_skip} dev={dv_skip} | labels={label_map}")

    tokenizer = AutoTokenizer.from_pretrained(args.model_name)
    collate = make_collate(tokenizer, args.max_seq_length)
    micro_bs = args.batch_size // args.grad_accum
    gen = torch.Generator().manual_seed(args.seed)
    train_loader = DataLoader(train_ex, batch_size=micro_bs, shuffle=True, collate_fn=collate, generator=gen)
    dev_loader = DataLoader(dev_ex, batch_size=args.eval_batch_size, shuffle=False, collate_fn=collate)

    # ---- model, optimizer, schedule ------------------------------------------
    model = AutoModelForSequenceClassification.from_pretrained(args.model_name, num_labels=n_labels).to(device)
    no_decay = ("bias", "LayerNorm.weight")
    groups = [
        {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
         "weight_decay": args.weight_decay},
        {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
         "weight_decay": 0.0},
    ]
    optimizer = torch.optim.AdamW(groups, lr=args.learning_rate, eps=1e-8)
    steps_per_epoch = math.ceil(len(train_loader) / args.grad_accum)
    total_steps = steps_per_epoch * args.epochs
    warmup_steps = int(args.warmup_ratio * total_steps)

    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = make_scaler(use_fp16)

    # ---- training -------------------------------------------------------------
    print(f"training on {gpu_name or 'CPU'} | {args.epochs} epochs x {steps_per_epoch} steps | "
          f"batch {args.batch_size} (micro-batch {micro_bs}) | lr {args.learning_rate} | fp16={use_fp16}")
    history, global_step = [], 0
    model.train()
    for epoch in range(1, args.epochs + 1):
        running_loss, n_loss = 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        for i, batch in enumerate(train_loader, start=1):
            batch = {k: v.to(device) for k, v in batch.items()}
            with amp(use_fp16):
                loss = model(**batch).loss / args.grad_accum
            scaler.scale(loss).backward()
            running_loss += loss.item() * args.grad_accum
            n_loss += 1
            if i % args.grad_accum == 0 or i == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), args.max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1
                if global_step % args.log_every == 0:
                    print(f"  epoch {epoch} step {global_step}/{total_steps} | loss {running_loss / n_loss:.4f} "
                          f"| lr {scheduler.get_last_lr()[0]:.2e} | {(time.time() - t_start) / 60:.1f} min")
        dev_metrics, preds = evaluate(model, dev_loader, device, use_fp16, n_labels)
        history.append({"epoch": epoch, "train_loss": running_loss / max(1, n_loss), "dev": dev_metrics,
                        "elapsed_min": round((time.time() - t_start) / 60, 2)})
        print(f"epoch {epoch} done | train loss {running_loss / max(1, n_loss):.4f} | dev {dev_metrics}")

    # ---- results (final = after the last epoch, as the original run_glue.py reports) ----
    final_metrics = history[-1]["dev"]
    run_name = args.run_name or "_".join(
        [p for p in os.path.normpath(os.path.abspath(args.data_dir)).split(os.sep) if p][-3:])
    out_dir = os.path.join(args.output_root, run_name)
    os.makedirs(out_dir, exist_ok=True)
    id2label = {v: k for k, v in label_map.items()}
    with open(os.path.join(out_dir, "dev_predictions.tsv"), "w", encoding="utf-8") as f:
        f.write("text_a\ttext_b\tgold\tpred\n")
        for (a, b, gold), pred in zip(dev_ex, preds):
            f.write(f"{a}\t{b or ''}\t{id2label[gold]}\t{id2label[pred]}\n")
    runtime_min = round((time.time() - t_start) / 60, 2)
    report = {
        "run_name": run_name,
        "data_dir": os.path.abspath(args.data_dir),
        "format": fmt,
        "label_map": label_map,
        "n_train": len(train_ex),
        "n_dev": len(dev_ex),
        "skipped_rows": {"train": tr_skip, "dev": dv_skip},
        "settings": vars(args),
        "environment": {
            "python": platform.python_version(),
            "torch": torch.__version__,
            "transformers": transformers.__version__,
            "device": gpu_name or "CPU",
            "platform": platform.platform(),
        },
        "epochs": history,
        "final_dev": final_metrics,
        "runtime_min": runtime_min,
    }
    with open(os.path.join(out_dir, "metrics.json"), "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)
    if args.save_model:
        model.save_pretrained(os.path.join(out_dir, "model"))
        tokenizer.save_pretrained(os.path.join(out_dir, "model"))
    summary = " ".join(f"{k}={v:.4f}" for k, v in final_metrics.items())
    print(f"RESULT {run_name} | {summary} | n_dev={len(dev_ex)} | {runtime_min} min on {gpu_name or 'CPU'}")
    print(f"saved to {out_dir}/")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile build_reviews_dataset.py
#!/usr/bin/env python3
"""
build_reviews_dataset.py: builds the project's newly constructed dataset,
"AU health & banking app reviews", for the SanText replication (COMP8240).

Why this data: SanText was motivated by health and finance text. App-store
reviews of health and banking apps often disclose exactly that kind of
information (conditions, medications, banks, money problems, family), and each
review's star rating gives a sentiment label for free, as in IMDb (Maas et al., 2011).

Stages (run in order, or all at once with `all`):
  collect     Newest Google Play reviews for each app in APPS (country=au,
              lang=en) via the google-play-scraper package. Reviewer names,
              avatars and developer replies are never stored.
  preprocess  clean -> drop duplicates -> keep >= 5 words -> keep English ->
              star label (1-2 stars = 0 negative, 4-5 = 1 positive, 3 dropped) ->
              stratified 80/20 train/dev split -> SST-2-format TSVs that
              run_SanText.py --task SST-2 and finetune_sanitized.py read unchanged.
              Writes stats.json with the stage-by-stage funnel (for the update video).
  sample      Stratified sample of reviews for manual annotation (CSV to fill in).

Usage
  pip install google-play-scraper          # only needed for `collect`
  python build_reviews_dataset.py collect --per_app 300
  python build_reviews_dataset.py preprocess
  python build_reviews_dataset.py sample --n 200

Outputs (under --out_dir, default new_data/app_reviews/)
  raw_reviews.jsonl, collection_log.json     collect
  reviews_clean.jsonl, train.tsv, dev.tsv, stats.json   preprocess
  annotation/annotation_sheet.csv            sample
"""
import argparse
import csv
import hashlib
import json
import os
import random
import re
import statistics
import time
from collections import Counter, defaultdict
from datetime import datetime, timezone

# (Google Play package id, category, display name).
# Check each id against the app's Play Store URL (...details?id=<package id>)
# and edit freely; unknown ids are reported and skipped.
APPS = [
    ("com.mysugr.android.companion", "health", "mySugr (diabetes)"),
    ("com.medisafe.android.client", "health", "Medisafe (medications)"),
    ("eu.smartpatient.mytherapy", "health", "MyTherapy (medications)"),
    ("com.getsomeheadspace.android", "health", "Headspace (mental health)"),
    ("com.commbank.netbank", "finance", "CommBank"),
    ("org.westpac.bank", "finance", "Westpac"),
    ("au.com.nab.mobile", "finance", "NAB"),
    ("com.anz.android.gomoney", "finance", "ANZ"),
]

# Common English function words; a review needs at least two to count as English.
EN_HINTS = set(
    "the and to is it i this my of for a in not you with on have but was be can "
    "so me that when are just very all no an at or its it's app".split()
)
WS = re.compile(r"\s+")  # also matches tabs and newlines, which would break the TSV format
WORD = re.compile(r"[a-z']+")


# ----------------------------------------------------------------------------- collect
def collect(args):
    try:
        from google_play_scraper import Sort, reviews
    except ImportError:
        raise SystemExit("google-play-scraper is not installed: run  pip install google-play-scraper")
    os.makedirs(args.out_dir, exist_ok=True)
    raw_path = os.path.join(args.out_dir, "raw_reviews.jsonl")
    log = {"collected_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
           "country": args.country, "lang": args.lang, "per_app_target": args.per_app, "apps": []}
    seen = set()
    with open(raw_path, "w", encoding="utf-8") as out:
        for app_id, category, name in APPS:
            got, token, error = 0, None, None
            try:
                while got < args.per_app:
                    batch, token = reviews(app_id, lang=args.lang, country=args.country, sort=Sort.NEWEST,
                                           count=min(200, args.per_app - got), continuation_token=token)
                    batch = [r for r in batch if r.get("reviewId") not in seen][: args.per_app - got]
                    if not batch:
                        break
                    for r in batch:
                        seen.add(r.get("reviewId"))
                        at = r.get("at")
                        record = {
                            "review_id": r.get("reviewId"),
                            "app_id": app_id,
                            "app_name": name,
                            "category": category,
                            "stars": r.get("score"),
                            "text": r.get("content") or "",
                            "date": at.isoformat() if hasattr(at, "isoformat") else at,
                            "app_version": r.get("appVersion") or r.get("reviewCreatedVersion"),
                            "thumbs_up": r.get("thumbsUpCount"),
                        }
                        out.write(json.dumps(record, ensure_ascii=False) + "\n")
                    got += len(batch)
                    if getattr(token, "token", None) is None:
                        break
                    time.sleep(args.sleep)
            except Exception as e:  # unknown package id, network problem, layout change
                error = f"{type(e).__name__}: {e}"
            log["apps"].append({"app_id": app_id, "app_name": name, "category": category,
                                "collected": got, "error": error})
            print(f"  {name:<30} {got:>5} reviews" + (f"   ! {error}" if error else ""))
    with open(os.path.join(args.out_dir, "collection_log.json"), "w", encoding="utf-8") as f:
        json.dump(log, f, indent=2)
    print(f"collected {sum(a['collected'] for a in log['apps'])} reviews -> {raw_path}")


# ----------------------------------------------------------------------------- preprocess
def clean_text(text):
    return WS.sub(" ", (text or "").replace("​", " ")).strip()


def looks_english(text):
    letters = [c for c in text if c.isalpha()]
    if not letters:
        return False
    ascii_share = sum(c.isascii() for c in letters) / len(letters)
    hits = sum(w in EN_HINTS for w in WORD.findall(text.lower()))
    return ascii_share >= 0.9 and hits >= 2


def star_label(stars):
    if stars in (1, 2):
        return 0
    if stars in (4, 5):
        return 1
    return None  # 3 stars (neutral) or missing


def preprocess(args):
    raw_path = os.path.join(args.out_dir, "raw_reviews.jsonl")
    with open(raw_path, encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]

    funnel = []  # (stage, kept rows)

    def record(stage, kept):
        per_cat = Counter(r["category"] for r in kept)
        funnel.append({"stage": stage, "total": len(kept), **{c: per_cat.get(c, 0) for c in ("health", "finance")}})
        return kept

    rows = record("collected", rows)
    for r in rows:
        r["text"] = clean_text(r.get("text"))
    rows = record("non-empty text", [r for r in rows if r["text"]])
    seen, unique = set(), []
    for r in rows:
        key = r["text"].lower()
        if key not in seen:
            seen.add(key)
            unique.append(r)
    rows = record("duplicates removed", unique)
    rows = record(f">= {args.min_words} words", [r for r in rows if len(r["text"].split()) >= args.min_words])
    rows = record("English only", [r for r in rows if looks_english(r["text"])])
    for r in rows:
        r["label"] = star_label(r.get("stars"))
    rows = record("3-star/unrated dropped", [r for r in rows if r["label"] is not None])

    # stratified split by (category, label) so both splits keep the same mix
    rng = random.Random(args.seed)
    groups = defaultdict(list)
    for r in rows:
        groups[(r["category"], r["label"])].append(r)
    for key in sorted(groups):
        members = groups[key]
        rng.shuffle(members)
        n_dev = round(len(members) * args.dev_share)
        for i, r in enumerate(members):
            r["split"] = "dev" if i < n_dev else "train"

    for split in ("train", "dev"):
        with open(os.path.join(args.out_dir, f"{split}.tsv"), "w", encoding="utf-8") as f:
            f.write("sentence\tlabel\n")
            for r in rows:
                if r["split"] == split:
                    f.write(f"{r['text']}\t{r['label']}\n")
    with open(os.path.join(args.out_dir, "reviews_clean.jsonl"), "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    lengths = [len(r["text"].split()) for r in rows]
    vocab = Counter(w for r in rows for w in WORD.findall(r["text"].lower()))
    stats = {
        "funnel": funnel,
        "final": {
            "reviews": len(rows),
            "train": sum(r["split"] == "train" for r in rows),
            "dev": sum(r["split"] == "dev" for r in rows),
            "share_positive": round(sum(r["label"] for r in rows) / max(1, len(rows)), 3),
            "by_category_and_label": {f"{c}_{'pos' if l else 'neg'}": len(g) for (c, l), g in sorted(groups.items())},
            "by_app": dict(Counter(r["app_name"] for r in rows)),
            "words_per_review": {
                "mean": round(statistics.mean(lengths), 1) if lengths else 0,
                "median": statistics.median(lengths) if lengths else 0,
                "max": max(lengths, default=0),
            },
            "word_types": len(vocab),
            # SanText builds a |V| x |V| matrix; peak memory measured at ~5 copies of it (float32)
            "santext_peak_memory_gb_upper_bound": round(5 * len(vocab) ** 2 * 4 / 1e9, 2),
        },
    }
    with open(os.path.join(args.out_dir, "stats.json"), "w", encoding="utf-8") as f:
        json.dump(stats, f, indent=2)

    print(f"{'stage':<26}{'total':>7}{'health':>8}{'finance':>9}")
    for s in funnel:
        print(f"{s['stage']:<26}{s['total']:>7}{s['health']:>8}{s['finance']:>9}")
    fin = stats["final"]
    print(f"\ntrain {fin['train']} | dev {fin['dev']} | positive share {fin['share_positive']} | "
          f"words/review mean {fin['words_per_review']['mean']} median {fin['words_per_review']['median']} | "
          f"word types {fin['word_types']} (SanText peak memory <= {fin['santext_peak_memory_gb_upper_bound']} GB)")
    print(f"written: train.tsv, dev.tsv, reviews_clean.jsonl, stats.json in {args.out_dir}")


# ----------------------------------------------------------------------------- sample
SHEET_COLUMNS = [
    "ann_id", "review_key", "category", "app_name", "stars", "star_label", "text",
    "text_sentiment",          # positive / negative / mixed   (judge the words, ignore the stars)
    "discloses_personal_info",  # yes / no
    "sensitive_words",         # exact words from the text, separated by ;
    "sensitive_categories",    # HEALTH / FINANCE / LOCATION / IDENTITY / CONTACT, separated by ;
    "notes",
]


def sample(args):
    with open(os.path.join(args.out_dir, "reviews_clean.jsonl"), encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]
    groups = defaultdict(list)
    for r in rows:
        groups[(r["category"], r["label"])].append(r)
    keys = sorted(groups)
    rng = random.Random(args.seed)
    for k in keys:
        rng.shuffle(groups[k])
    # equal share per (category, label) cell; leftover quota goes to cells that still have rows
    quota = {k: 0 for k in keys}
    remaining = args.n
    while remaining > 0 and any(quota[k] < len(groups[k]) for k in keys):
        for k in keys:
            if remaining and quota[k] < len(groups[k]):
                quota[k] += 1
                remaining -= 1
    picked = [r for k in keys for r in groups[k][: quota[k]]]
    rng.shuffle(picked)  # mix categories so the annotator can't infer the stars from position

    os.makedirs(os.path.join(args.out_dir, "annotation"), exist_ok=True)
    path = os.path.join(args.out_dir, "annotation", "annotation_sheet.csv")
    with open(path, "w", encoding="utf-8-sig", newline="") as f:  # utf-8-sig opens cleanly in Excel
        w = csv.DictWriter(f, fieldnames=SHEET_COLUMNS)
        w.writeheader()
        for i, r in enumerate(picked, start=1):
            key = hashlib.sha1((r.get("review_id") or r["text"]).encode("utf-8")).hexdigest()[:12]
            w.writerow({"ann_id": i, "review_key": key, "category": r["category"], "app_name": r["app_name"],
                        "stars": r["stars"], "star_label": "positive" if r["label"] else "negative",
                        "text": r["text"]})
    cells = ", ".join(f"{c}/{'pos' if l else 'neg'}={quota[(c, l)]}" for c, l in keys)
    print(f"annotation sheet with {len(picked)} reviews ({cells}) -> {path}")


# ----------------------------------------------------------------------------- main
def main():
    p = argparse.ArgumentParser(description="Build the AU health & banking app-review dataset.")
    p.add_argument("stage", choices=["collect", "preprocess", "sample", "all"])
    p.add_argument("--out_dir", default="new_data/app_reviews")
    p.add_argument("--per_app", type=int, default=300, help="reviews to collect per app")
    p.add_argument("--country", default="au")
    p.add_argument("--lang", default="en")
    p.add_argument("--sleep", type=float, default=1.0, help="pause between requests, in seconds")
    p.add_argument("--min_words", type=int, default=5)
    p.add_argument("--dev_share", type=float, default=0.2)
    p.add_argument("--n", type=int, default=200, help="reviews in the annotation sample")
    p.add_argument("--seed", type=int, default=42)
    args = p.parse_args()
    if args.stage in ("collect", "all"):
        collect(args)
    if args.stage in ("preprocess", "all"):
        preprocess(args)
    if args.stage in ("sample", "all"):
        sample(args)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile measure.py
"""Run a command, then print its wall time and peak memory (resident set size)."""
import resource
import subprocess
import sys
import time

t0 = time.time()
code = subprocess.run(sys.argv[1:]).returncode
peak_gb = resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss * 1024 / 1e9  # ru_maxrss is in KiB on Linux
print(f"[measure] exit={code} wall={time.time() - t0:.0f}s peak_memory={peak_gb:.2f} GB", flush=True)
sys.exit(code)

In [ ]:
# Step 1 · Data: SST-2 now, GloVe (2.2 GB) in the background
DATA = f"{BASE}/SanText/data"
os.makedirs(DATA, exist_ok=True)
if not os.path.exists(f"{DATA}/SST-2/train.tsv"):
    !cd {DATA} && wget -q -O SST-2.zip https://dl.fbaipublicfiles.com/glue/data/SST-2.zip && unzip -o -q SST-2.zip && rm SST-2.zip
!wc -l {DATA}/SST-2/train.tsv {DATA}/SST-2/dev.tsv
!head -3 {DATA}/SST-2/train.tsv

GLOVE_TXT = f"{DATA}/glove.840B.300d.txt"
GLOVE_URLS = ["https://huggingface.co/stanfordnlp/glove/resolve/main/glove.840B.300d.zip",  # mirror, usually faster
              "https://nlp.stanford.edu/data/glove.840B.300d.zip"]                         # original
glove_proc = None
if not os.path.exists(GLOVE_TXT):
    glove_cmd = (" || ".join(f"wget -q -O {DATA}/glove.zip {u}" for u in GLOVE_URLS)
                 + f" && unzip -o -q {DATA}/glove.zip -d {DATA} && rm -f {DATA}/glove.zip")
    glove_proc = subprocess.Popen(["bash", "-c", glove_cmd], stdout=open(f"{BASE}/logs/glove_download.log", "w"),
                                  stderr=subprocess.STDOUT)
    print("GloVe download started in the background; Step 4 waits for it.")

In [ ]:
# Step 2 · Constructed dataset: AU health & banking app reviews (about 5 minutes)
!cd {BASE} && python -u build_reviews_dataset.py collect --per_app 500 2>&1 | tee logs/reviews_collect.log
!cd {BASE} && python -u build_reviews_dataset.py preprocess 2>&1 | tee logs/reviews_preprocess.log
!cd {BASE} && python -u build_reviews_dataset.py sample --n 100 2>&1 | tee logs/reviews_sample.log
backup(f"{BASE}/new_data", f"{BASE}/logs")
if os.path.exists(f"{BASE}/new_data/app_reviews/annotation/annotation_sheet.csv"):
    where = f"{BACKUP}/new_data/app_reviews/annotation/" if BACKUP else "the Files pane (folder icon): new_data/app_reviews/annotation/"
    print(f"\nAnnotation sheet ready: annotation_sheet.csv in {where}")

### Annotating the 100-review sample (about 50 minutes, while the GPU works)
Open `annotation_sheet.csv` in Google Sheets or Excel. Judge the **review text only** and ignore the stars. Fill four columns:

1. **`text_sentiment`**: `positive` / `negative` / `mixed`.
2. **`discloses_personal_info`**: `yes` / `no`. "Yes" means the review reveals something about the writer or an identifiable person that they might not want linked to them:
   - **HEALTH**: conditions, symptoms, medications, treatment, pregnancy, mental health
   - **FINANCE**: bank, amounts, debt, income, account problems
   - **LOCATION**: suburb, address, workplace, branch
   - **IDENTITY**: name, age, family members, job
   - **CONTACT**: email, phone, account or reference numbers

   Opinions about the app, its features or generic use ("I use it daily") are **not** disclosures.
3. **`sensitive_words`**: the exact words from the text, separated by `;` (e.g. `type 1 diabetic; insulin; my son`).
4. **`sensitive_categories`**: the category of each, separated by `;` (e.g. `HEALTH; HEALTH; IDENTITY`).

When unsure, choose `no` and say why in `notes`. Save as CSV and upload it in the chat.

In [ ]:
# Step 3 · Fine-tuning: 1-minute smoke test, then the unsanitized baseline (paper: 0.9251)
!cd {BASE} && python -u finetune_sanitized.py --data_dir SanText/data/SST-2 --max_train_samples 512 --epochs 1 --fp16 --run_name smoke_test 2>&1 | tee logs/ft_smoke_test.log
!cd {BASE} && python -u finetune_sanitized.py --data_dir SanText/data/SST-2 --fp16 --log_every 300 --run_name SST-2_unsanitized 2>&1 | tee logs/ft_SST-2_unsanitized.log
backup(f"{BASE}/results", f"{BASE}/logs")

In [ ]:
# Step 4 · SanText sanitization with the authors' unchanged code (GloVe, ε = 1, 2, 3)
t0 = time.time()
while glove_proc is not None and glove_proc.poll() is None:
    zip_path = f"{DATA}/glove.zip"
    got = os.path.getsize(zip_path) / 1e9 if os.path.exists(zip_path) else 0.0
    print(f"waiting for GloVe: {got:.2f} of 2.18 GB downloaded, then unzipping ({(time.time() - t0) / 60:.1f} min)", flush=True)
    time.sleep(30)
if not os.path.exists(GLOVE_TXT):
    raise RuntimeError("GloVe is missing; see logs/glove_download.log")
print("GloVe ready:", round(os.path.getsize(GLOVE_TXT) / 1e9, 2), "GB")

for eps in ["1.0", "2.0", "3.0"]:
    !cd {BASE}/SanText && python -u {BASE}/measure.py python -u run_SanText.py --task SST-2 --method SanText --epsilon {eps} --word_embedding_path ./data/glove.840B.300d.txt --word_embedding_size 300 --data_dir ./data/SST-2/ --output_dir ./output_SanText_glove/SST-2/ --threads 8 2>&1 | tee {BASE}/logs/santext_eps_{eps}.log
backup(f"{BASE}/SanText/output_SanText_glove", f"{BASE}/logs")

In [ ]:
# Step 5 · Fine-tune on the sanitized data (paper: ε = 1 → 0.5101, ε = 2 → 0.5838, ε = 3 → 0.8374)
for e in ["1.00", "2.00", "3.00"]:
    !cd {BASE} && python -u finetune_sanitized.py --data_dir SanText/output_SanText_glove/SST-2/eps_{e} --fp16 --log_every 300 --run_name SST-2_SanText_eps_{e} 2>&1 | tee logs/ft_SST-2_SanText_eps_{e}.log
backup(f"{BASE}/results", f"{BASE}/logs")

In [ ]:
# Step 6 · Results vs the paper, memory, and what sanitization does to the text
import unicodedata
from spacy.lang.en import English

out = ["# Core results", ""]

# 1) Utility: dev accuracy vs Yue et al. (2021), Table 1 (GloVe, SanText)
PAPER = [("SST-2_unsanitized", "Unsanitized", 0.9251), ("SST-2_SanText_eps_1.00", "SanText ε = 1", 0.5101),
         ("SST-2_SanText_eps_2.00", "SanText ε = 2", 0.5838), ("SST-2_SanText_eps_3.00", "SanText ε = 3", 0.8374)]
out += ["## Utility (SST-2 dev, n = 872)", "",
        "| Setting | Ours: accuracy | Ours: macro-F1 | Paper: accuracy | Difference | Minutes |", "|---|---|---|---|---|---|"]
env = None
for run, label, paper in PAPER:
    p = f"{BASE}/results/{run}/metrics.json"
    if os.path.exists(p):
        m = json.load(open(p))
        env = env or m["environment"]
        acc, f1 = m["final_dev"]["accuracy"], m["final_dev"]["macro_f1"]
        out.append(f"| {label} | {acc:.4f} | {f1:.4f} | {paper:.4f} | {acc - paper:+.4f} | {m['runtime_min']} |")
    else:
        out.append(f"| {label} | missing | missing | {paper:.4f} | | |")
if env:
    out += ["", f"Environment: {env['device']}, Python {env['python']}, torch {env['torch']}, transformers {env['transformers']}"]

# 2) Vocabulary and memory, from the sanitization logs
out += ["", "## Vocabulary and memory (sanitization step)", "",
        "| ε | Vocabulary (paper: 14,730) | With GloVe vectors | Peak memory | Wall time |", "|---|---|---|---|---|"]
peaks, n_matrix = [], None
for eps in ["1.0", "2.0", "3.0"]:
    p = f"{BASE}/logs/santext_eps_{eps}.log"
    log = open(p, errors="ignore").read() if os.path.exists(p) else ""
    total = re.search(r"#Total Words: (\d+)", log)
    emb = re.search(r"All Word Embedding Matrix: \((\d+), (\d+)\)", log)
    meas = re.search(r"\[measure\] exit=(\d+) wall=(\d+)s peak_memory=([\d.]+) GB", log)
    if emb:
        n_matrix = int(emb.group(1))
    if meas:
        peaks.append(float(meas.group(3)))
    out.append(f"| {eps} | {total.group(1) if total else '?'} | {emb.group(1) if emb else '?'} | "
               f"{meas.group(3) + ' GB' if meas else '?'} | {meas.group(2) + ' s' if meas else '?'} |")
if peaks and n_matrix:
    qnli = max(peaks) * (88159 / n_matrix) ** 2
    out += ["", f"Scaling the peak by vocabulary size squared, QNLI (88,159 words) would need about {qnli:.0f} GB."]

# 3) How much text survives: share of tokens left unchanged (dev set)
nlp = English()
def tsv_texts(path):
    with open(path, encoding="utf-8") as f:
        return [line.rstrip("\n").split("\t")[0] for line in f][1:]
orig = tsv_texts(f"{DATA}/SST-2/dev.tsv")
orig_tok = [[unicodedata.normalize("NFD", t.text) for t in nlp(s)] for s in orig]
san = {}
out += ["", "## Tokens left unchanged by sanitization (dev set)", "", "| ε | Unchanged tokens | Sentences compared |", "|---|---|---|"]
for e in ["1.00", "2.00", "3.00"]:
    p = f"{BASE}/SanText/output_SanText_glove/SST-2/eps_{e}/dev.tsv"
    if not os.path.exists(p):
        continue
    san[e] = tsv_texts(p)
    same = total = used = 0
    for o, s in zip(orig_tok, san[e]):
        st = s.split(" ")
        if len(o) != len(st):
            continue
        used += 1
        total += len(o)
        same += sum(a == b for a, b in zip(o, st))
    out.append(f"| {float(e):g} | {same / max(1, total):.1%} | {used} of {len(orig)} |")

# 4) Three examples, original vs sanitized
out += ["", "## Examples (dev set)", ""]
picked = [i for i, toks in enumerate(orig_tok) if 6 <= len(toks) <= 12][:3]
for i in picked:
    out.append(f"- original: {orig[i]}")
    for e in sorted(san):
        out.append(f"  - ε = {float(e):g}: {san[e][i]}")

summary = "\n".join(out)
print(summary)
open(f"{BASE}/results/summary.md", "w", encoding="utf-8").write(summary)
!cd {BASE} && zip -qr core_results.zip results logs new_data/app_reviews/stats.json new_data/app_reviews/collection_log.json
backup(f"{BASE}/results", f"{BASE}/logs", f"{BASE}/core_results.zip")
print("\ncore_results.zip is ready (Files pane, or the Drive folder).")

## Optional extras
These run automatically after the core steps. If you're short of time, stop here (**Runtime → Interrupt execution**) and upload `core_results.zip`.

In [ ]:
# Step 7 · Optional: Dreaddit (existing dataset): statistics, then unsanitized vs SanText ε = 3
import csv, statistics, zipfile, urllib.request
DD = f"{BASE}/new_data/dreaddit"
os.makedirs(DD, exist_ok=True)
DREADDIT_OK = False
try:
    if not glob.glob(f"{DD}/**/dreaddit-train.csv", recursive=True):
        urllib.request.urlretrieve("http://www.cs.columbia.edu/~eturcan/data/dreaddit.zip", f"{DD}/dreaddit.zip")
        with zipfile.ZipFile(f"{DD}/dreaddit.zip") as z:
            z.extractall(DD)
    stats = {}
    for split, name in [("train", "dreaddit-train.csv"), ("dev", "dreaddit-test.csv")]:
        src = glob.glob(f"{DD}/**/{name}", recursive=True)[0]
        with open(src, encoding="utf-8", newline="") as f:
            rows = [r for r in csv.DictReader(f) if r.get("text", "").strip() and r.get("label", "").strip() in ("0", "1")]
        with open(f"{DD}/{split}.tsv", "w", encoding="utf-8") as f:
            f.write("sentence\tlabel\n")
            for r in rows:
                f.write(" ".join(r["text"].split()) + "\t" + r["label"].strip() + "\n")
        lengths = [len(r["text"].split()) for r in rows]
        stats[split] = {"rows": len(rows),
                        "share_stress": round(sum(r["label"].strip() == "1" for r in rows) / len(rows), 3),
                        "mean_words": round(statistics.mean(lengths), 1), "max_words": max(lengths),
                        "subreddits": len({r.get("subreddit") for r in rows})}
    json.dump(stats, open(f"{DD}/stats.json", "w"), indent=2)
    print(json.dumps(stats, indent=2))
    DREADDIT_OK = True
except Exception as e:
    print("Dreaddit skipped (optional):", e)

if DREADDIT_OK:
    !cd {BASE} && python -u finetune_sanitized.py --data_dir new_data/dreaddit --max_seq_length 256 --batch_size 16 --fp16 --run_name dreaddit_unsanitized 2>&1 | tee logs/ft_dreaddit_unsanitized.log
    !cd {BASE}/SanText && python -u {BASE}/measure.py python -u run_SanText.py --task SST-2 --method SanText --epsilon 3.0 --word_embedding_path ./data/glove.840B.300d.txt --word_embedding_size 300 --data_dir {DD}/ --output_dir ./output_SanText_glove/dreaddit/ --threads 8 2>&1 | tee {BASE}/logs/santext_dreaddit_eps_3.0.log
    !cd {BASE} && python -u finetune_sanitized.py --data_dir SanText/output_SanText_glove/dreaddit/eps_3.00 --max_seq_length 256 --batch_size 16 --fp16 --run_name dreaddit_SanText_eps_3.00 2>&1 | tee logs/ft_dreaddit_SanText_eps_3.00.log
    backup(f"{BASE}/results", f"{BASE}/logs", f"{DD}/stats.json")

In [ ]:
# Step 8 · Optional: first SanText results on the constructed review dataset (ε = 3)
RV = f"{BASE}/new_data/app_reviews"
if os.path.exists(f"{RV}/train.tsv"):
    !cd {BASE} && python -u finetune_sanitized.py --data_dir new_data/app_reviews --batch_size 16 --fp16 --run_name reviews_unsanitized 2>&1 | tee logs/ft_reviews_unsanitized.log
    !cd {BASE}/SanText && python -u {BASE}/measure.py python -u run_SanText.py --task SST-2 --method SanText --epsilon 3.0 --word_embedding_path ./data/glove.840B.300d.txt --word_embedding_size 300 --data_dir {RV}/ --output_dir ./output_SanText_glove/app_reviews/ --threads 8 2>&1 | tee {BASE}/logs/santext_reviews_eps_3.0.log
    !cd {BASE} && python -u finetune_sanitized.py --data_dir SanText/output_SanText_glove/app_reviews/eps_3.00 --batch_size 16 --fp16 --run_name reviews_SanText_eps_3.00 2>&1 | tee logs/ft_reviews_SanText_eps_3.00.log
    backup(f"{BASE}/results", f"{BASE}/logs")
else:
    print("Review dataset missing (Step 2 failed); skipped.")

In [ ]:
# Step 9 · All runs in one table, plus one zip to upload in the chat
rows = ["| Run | Train | Dev | Accuracy | Macro-F1 | F1 (label 1) | Minutes |", "|---|---|---|---|---|---|---|"]
for p in sorted(glob.glob(f"{BASE}/results/*/metrics.json")):
    m = json.load(open(p))
    if m["run_name"] == "smoke_test":
        continue
    d = m["final_dev"]
    f1_pos = d.get("f1_label_index_1")
    f1_txt = "" if f1_pos is None else f"{f1_pos:.4f}"
    rows.append(f"| {m['run_name']} | {m['n_train']} | {m['n_dev']} | {d['accuracy']:.4f} | {d['macro_f1']:.4f} | {f1_txt} | {m['runtime_min']} |")
table = "\n".join(rows)
print(table)
open(f"{BASE}/results/all_runs.md", "w", encoding="utf-8").write(table)
!cd {BASE} && zip -qr comp8240_update_outputs.zip results logs new_data/app_reviews/stats.json new_data/app_reviews/collection_log.json new_data/app_reviews/annotation new_data/dreaddit/stats.json SanText/output_SanText_glove -x "*.jsonl"
!ls -lh {BASE}/comp8240_update_outputs.zip
backup(f"{BASE}/comp8240_update_outputs.zip")
print("Upload comp8240_update_outputs.zip in the chat (Files pane, or the Drive folder).")